# Chapter 3: Moduli Spaces and Transversality

    **Source orientation:** McDuff-Salamon, *J-holomorphic Curves and Symplectic Topology*, Chapter 3, printed pp. 39-74; PDF pp. 54-89. Sections: 3.1-3.5.

    ## Chapter Goal

    Moduli spaces of simple curves, transversality, regularity criteria, point constraints, and the implicit function theorem.

    The guiding question is:

    > When does the solution set of the nonlinear Cauchy-Riemann equation behave like a manifold of the predicted dimension?

    ## Computational Translation Guide

    | Chapter language | Computational object | Inspection target / check |
| --- | --- | --- |
| linearized operator | matrix with finite-dimensional kernel/cokernel | rank and index |
| regular curve | surjective linearization | cokernel dimension zero |
| moduli dimension | Fredholm index minus automorphisms | dimension ledger |
| point constraint | evaluation map meeting a cycle | codimension subtraction |
| implicit function theorem | zero set of a submersion | local chart residual |


## Standalone Reading Guide

    This chapter is where the equation becomes a geometric space. A moduli problem begins with a section of a Banach bundle: the section assigns to a map its Cauchy-Riemann defect. The desired curves are the zeros of that section. The linearization at a zero is a Fredholm operator, so it has a finite index even though it acts on infinite-dimensional function spaces.

Regularity is the demand that the linearized operator have no cokernel. Under that condition, the implicit function theorem identifies the nearby zero set with the kernel of the linearization. The expected dimension is therefore not a slogan; it is the dimension of a finite-dimensional tangent model corrected for automorphisms and constraints. Generic transversality says that, after allowing the almost complex structure or perturbation data to vary, regularity is not exceptional.

The finite-dimensional experiment below uses the familiar example of rational curves in the projective plane as a dimension ledger. A degree d curve carries a Chern-number contribution that grows with d. Marked-point constraints subtract dimension. The special number 3d-1 appears because it balances the real dimension to zero, the setting in which a signed count can become an invariant.

    ## Topics In This Notebook

    - Banach-manifold setup for maps and almost complex structures
- linearized Cauchy-Riemann operators and Fredholm index
- regular values, surjectivity, and generic transversality
- pointwise constraints as evaluation-map conditions
- implicit function theorem as the local model for moduli spaces

    ## Visualization Storyboard

    - A dimension plot for rational curves in CP2 shows why 3d-1 point conditions produce a zero-dimensional count.
- A dependency graph separates analytic surjectivity from geometric incidence.
- A ledger checks rank, cokernel, and expected dimensions before and after constraints.


In [ ]:
from pathlib import Path
import csv
import json
import math
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import sympy as sp


def find_book_root(start=None):
    start = (start or Path.cwd()).resolve()
    for base in [start, *start.parents]:
        for candidate in [base, base / "J-Holomorphic-Curves-and-Symplectic-Topology"]:
            if (candidate / "AGENTS.md").exists() and (candidate / "utils").exists():
                return candidate
    raise RuntimeError("JHCST book root not found")


BOOK_ROOT = find_book_root()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import assert_artifact, display_artifact, save_json, save_matplotlib

UNIT = 'chapter-03'
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / UNIT
FIG_DIR = ARTIFACT_ROOT / "figures"
CHECK_DIR = ARTIFACT_ROOT / "checks"
TABLE_DIR = ARTIFACT_ROOT / "tables"
HTML_DIR = ARTIFACT_ROOT / "html"
for folder in [FIG_DIR, CHECK_DIR, TABLE_DIR, HTML_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

BOOK_ROOT


## Proof Visual: Dependency Map

The graph below is a compact proof-state diagram. Read an arrow as "this idea must be under control before the next one can be used." The point is not to replace the analysis with a graph, but to keep the logical dependencies inspectable while the chapter moves between local equations, moduli spaces, compactness, and algebraic counts.


In [ ]:
CONCEPT_NODES = ['linearized operator', 'regular curve', 'moduli dimension', 'point constraint', 'implicit function theorem']
CONCEPT_EDGES = [('linearized operator', 'regular curve'), ('regular curve', 'moduli dimension'), ('moduli dimension', 'point constraint'), ('point constraint', 'implicit function theorem')]

G = nx.DiGraph()
G.add_nodes_from(CONCEPT_NODES)
G.add_edges_from(CONCEPT_EDGES)
pos = nx.spring_layout(G, seed=42, k=1.35)

fig, ax = plt.subplots(figsize=(9.5, 5.8))
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=18, width=1.8, edge_color="#435466")
nx.draw_networkx_nodes(G, pos, ax=ax, node_size=2300, node_color="#eaf2f8", edgecolors="#1f567d", linewidths=1.5)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=8.5, font_weight="bold")
ax.set_title('Proof dependency map: Moduli Spaces and Transversality')
ax.set_axis_off()
graph_path = save_matplotlib(fig, UNIT, "figures", "proof-dependency-map.png")
plt.close(fig)

graph_check = {
    "nodes": len(CONCEPT_NODES),
    "edges": len(CONCEPT_EDGES),
    "is_directed_acyclic_graph": nx.is_directed_acyclic_graph(G),
    "source_span": '39-74',
    "passed": len(CONCEPT_NODES) >= 5 and nx.is_directed_acyclic_graph(G),
}
graph_json = save_json(graph_check, UNIT, "checks", "proof-dependency-map.json")
display_artifact(graph_path, width=780)
graph_check


## Executable Model

This section builds a small model for one core mechanism in Moduli Spaces and Transversality. The model is intentionally finite and inspectable: it creates an artifact, records a JSON check, and gives a learner a parameter to perturb in the applied lab.


In [ ]:
degrees = np.arange(1, 7)
constraints = 3 * degrees - 1
unconstrained_dim = 6 * degrees - 2
constrained_dim = unconstrained_dim - 2 * constraints
rank_model = np.array([[1.0, 0.2, 0.1], [0.0, 1.0, 0.3], [0.1, -0.2, 1.0], [0.3, 0.4, 0.2]])
rank = int(np.linalg.matrix_rank(rank_model))
cokernel = rank_model.shape[0] - rank

fig, ax = plt.subplots(figsize=(7.2, 4.8))
ax.plot(degrees, unconstrained_dim, marker="o", label="unconstrained real dimension")
ax.plot(degrees, 2 * constraints, marker="s", label="point-constraint codimension")
ax.plot(degrees, constrained_dim, marker="^", label="remaining dimension")
ax.axhline(0, color="black", linewidth=1)
ax.set_xlabel("degree d in CP2 model")
ax.set_ylabel("real dimension")
ax.set_title("Dimension balance for rational curves through points")
ax.legend()
ax.grid(True, alpha=0.25)
fig_path = save_matplotlib(fig, UNIT, "figures", "transversality-dimension-balance.png")
plt.close(fig)

check = {
    "degrees": degrees.tolist(),
    "constraints_3d_minus_1": constraints.tolist(),
    "remaining_dimensions": constrained_dim.tolist(),
    "sample_linearized_rank": rank,
    "sample_cokernel_dimension": cokernel,
    "passed": bool(np.all(constrained_dim == 0) and cokernel == 1),
}
check_path = save_json(check, UNIT, "checks", "transversality-dimension-checks.json")
display_artifact(fig_path, width=680)
check


## Invariant Ledger

The ledger records the chapter vocabulary as computational objects plus explicit checks. It is a small source map inside the notebook: every row names what should be inspected when the figure or experiment is changed.


In [ ]:
ledger_rows = [{'item': 'linearized operator', 'computational_object': 'matrix with finite-dimensional kernel/cokernel', 'check': 'rank and index'}, {'item': 'regular curve', 'computational_object': 'surjective linearization', 'check': 'cokernel dimension zero'}, {'item': 'moduli dimension', 'computational_object': 'Fredholm index minus automorphisms', 'check': 'dimension ledger'}, {'item': 'point constraint', 'computational_object': 'evaluation map meeting a cycle', 'check': 'codimension subtraction'}, {'item': 'implicit function theorem', 'computational_object': 'zero set of a submersion', 'check': 'local chart residual'}]
table_path = TABLE_DIR / "invariant-ledger.csv"
with table_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=["item", "computational_object", "check"])
    writer.writeheader()
    writer.writerows(ledger_rows)

ledger_check = {
    "row_count": len(ledger_rows),
    "items": [row["item"] for row in ledger_rows],
    "has_source_specific_checks": all(row["check"] for row in ledger_rows),
    "passed": len(ledger_rows) >= 5 and all(row["check"] for row in ledger_rows),
}
ledger_json = save_json(ledger_check, UNIT, "checks", "invariant-ledger.json")
display_artifact(table_path)
ledger_check


## Applied Lab

Change the number of point constraints away from 3d-1. The ledger will show positive-dimensional families or overdetermined systems instead of a count.

The intended workflow is to change one parameter, rerun the executable model, and then inspect both the figure and JSON check. If the visual impression and the invariant check disagree, trust the check first and then ask what the visualization is hiding.


## Takeaways

    - Transversality is the bridge from equation solving to manifold geometry.
- The Fredholm index is the source of the expected dimension.
- Point constraints should be read as codimension bookkeeping through evaluation maps.

    ## Sanity Checks

    The final cell asserts that the generated figures, ledgers, and JSON checks exist, are nonempty, and report successful chapter-specific invariants.


In [ ]:
expected = [
    FIG_DIR / "proof-dependency-map.png",
    FIG_DIR / 'transversality-dimension-balance.png',
    CHECK_DIR / "proof-dependency-map.json",
    CHECK_DIR / 'transversality-dimension-checks.json',
    CHECK_DIR / "invariant-ledger.json",
    TABLE_DIR / "invariant-ledger.csv",
]
for path in expected:
    min_bytes = 80 if path.suffix == ".csv" else 512
    assert_artifact(path, min_bytes=min_bytes)

for path in [CHECK_DIR / "proof-dependency-map.json", CHECK_DIR / 'transversality-dimension-checks.json', CHECK_DIR / "invariant-ledger.json"]:
    data = json.loads(path.read_text(encoding="utf-8"))
    assert data.get("passed") is True, path

print(f"Validated {len(expected)} artifacts for {UNIT}")
